In [39]:

from config import timer, fmp_update_endpoints, schema_map, ROOT, engine
import pandas as pd 
import os 
from dotenv import load_dotenv
import glob
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta

In [40]:
pd.read_sql('Select * from companies', con = engine)

,ticker,company_name,price,market_cap,beta,sector,industry,country,cik,isin,cusip,exchange,ceo,full_time_employees,ipo_date,description
0,0P00000SXJ,Franklin Technology A Acc USD,57.8900,8.432643e+08,1.080,Financial Services,Asset Management,LU,NaN,LU0109392836,NaN,OTC,NaN,NaN,2022-03-14,Franklin Covey Co. provides training and consu...
1,A,"Agilent Technologies, Inc.",113.5100,3.210513e+10,1.312,Healthcare,Medical - Diagnostics & Research,US,1090872.0,US00846U1016,00846U101,NYSE,Padraig McDonnell,17900.0,1999-11-18,"Agilent Technologies, Inc. provides applicatio..."
2,AA,Alcoa Corporation,57.5900,1.519455e+10,1.779,Basic Materials,Aluminum,US,1675149.0,US0138721065,013872106,NYSE,William F. Oplinger,13900.0,2016-11-01,"Alcoa Corporation, together with its subsidiar..."
3,AABB,"Asia Broadband, Inc.",0.0146,6.771666e+07,1.568,Basic Materials,Industrial Materials,US,NaN,US04518L1008,04518L100,OTC,Chris Torres,42.0,1998-04-24,"Asia Broadband, Inc., through its subsidiary, ..."
4,AABPX,American Beacon Balanced Fund,11.4900,1.099334e+08,0.980,Financial Services,Asset Management,US,809593.0,US02368A8282,NaN,NASDAQ,NaN,NaN,1994-07-29,"Under normal circumstances, between 50% and 70..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17769,ZYXI,"Zynex, Inc.",0.1250,3.798579e+06,0.998,Healthcare,Medical - Distribution,US,846475.0,US98986M1036,98986M103,NASDAQ,Steven Lewis Dyson,1000.0,2004-02-25,"Zynex, Inc., through its subsidiaries, designs..."
17770,ZYXIQ,"Zynex, Inc.",0.0600,1.823316e+06,1.082,Healthcare,Medical - Distribution,US,846475.0,US98986M1036,98986M103,OTC,Steven Lewis Dyson,1000.0,2004-02-25,"Zynex, Inc., together with its subsidiaries, d..."
17771,ZZHGF,"ZhongAn Online P & C Insurance Co., Ltd.",1.5000,2.452219e+09,0.831,Financial Services,Insurance - Property & Casualty,CN,NaN,CNE100002QY7,NaN,OTC,Jiang Xing,2395.0,2017-12-28,"ZhongAn Online P & C Insurance Co., Ltd., an I..."
17772,ZZHGY,"ZhongAn Online P & C Insurance Co., Ltd.",1.9800,3.236924e+09,0.831,Financial Services,Insurance - Property & Casualty,CN,NaN,US98955F1057,NaN,OTC,Jiang Xing,2395.0,2021-03-31,"ZhongAn Online P & C Insurance Co., Ltd., an I..."


In [41]:
symbols = pd.read_sql('''
SELECT ticker from companies
Where country = "US"
'''
, con=engine)['ticker'].tolist()
symbols = symbols[:100]

In [42]:
date = pd.read_sql('''
SELECT date
FROM daily_prices
ORDER BY date DESC
LIMIT 1;
''', con=engine)['date'].astype('str').iloc[0]
date 

'2026-03-24'

In [43]:
@timer
def update_daily_data(symbols_lst, file_path):
    table = 'daily_prices'

    date = pd.read_sql(
        'SELECT MAX(date) as date FROM daily_prices',
        con=engine
    )['date'].iloc[0]

    next_day = (pd.to_datetime(date) + timedelta(days=1)).strftime('%Y-%m-%d')

    fetcher = YFinanceFetcher(
        symbols=symbols_lst,
        root=file_path,
        chunk_size=500
    )
    df = fetcher.fetch(start_date=next_day)

    if df.empty:
        print("No new data to insert.")
        return

    clean = Cleaner(root=file_path)
    clean.insert_to_sql(table=table, df=df)